> Superseded for the current experiment: use 02g_teachers_fe_jaah_pilot.ipynb for both F and E. This file preserves the earlier E-only setup.

# P2 · JAAH auxiliary-data pilot

Read docs/p2/jaah/P2_JAAH_EXPERIMENT.md first. This runs two matched 30-epoch jobs: a fresh Billboard E25 control and E25 with 248/990 batches replaced by JAAH each epoch. Separate 24/12-dimensional projections feed the same TCN and heads. F remains frozen. Enable Internet and a GPU. This is a single-seed development experiment, not a test-set result.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json
project = Path("/kaggle/working/chord-practice-system")
if not project.exists():
    subprocess.run(["git", "clone", "https://github.com/PhatLavar/chord-practice-system.git", str(project)], check=True)
os.chdir(project)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run([sys.executable,"-m","pip","install","--no-deps","-e","."],check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
import torch, numpy
print("torch", torch.__version__, "numpy", numpy.__version__, "CUDA", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Official inputs. Downloaded archives are read without extraction.
from urllib.request import urlretrieve
raw = project / "data/raw"
raw.mkdir(parents=True, exist_ok=True)
sources = {
    "index.csv": "https://www.dropbox.com/s/o0olz0uwl9z9stb/billboard-2.0-index.csv?dl=1",
    "full.tar.xz": "https://www.dropbox.com/s/t390alzrkx0c9yt/billboard-2.0.1-lab.tar.xz?dl=1",
    "billboard-2.0-chordino.tar.xz": "https://www.dropbox.com/s/e9dm23vbawg9dsw/billboard-2.0-chordino.tar.xz?dl=1",
}
for name, url in sources.items():
    if not (raw / name).exists():
        temporary = raw / (name + ".part")
        urlretrieve(url, temporary)
        temporary.replace(raw / name)
cache = project / "data/processed/p1"
if not (cache / "metadata.json").exists():
    subprocess.run([sys.executable, '-m', 'acr.data.billboard.prepare'], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], check=True)


In [ ]:
# Download the pinned official JAAH revision; keep generated audit files in runs/.
aux_cache = project / "data/processed/p2-jaah"
if not aux_cache.exists():
    subprocess.run([sys.executable, '-m', 'acr.data.jaah', "--download",
                    "--report", "runs/p2-jaah-preparation"], check=True)
# Validate all cache hashes and source identity against the committed audit.

from acr.experiments.jaah.event import load_aux
aux_songs, aux_events, aux_meta = load_aux(aux_cache)
print({"JAAH_recordings": len(aux_songs), "events": len(aux_events),
       "revision": aux_meta["revision"], "excluded": len(aux_meta["excluded"])})
del aux_songs, aux_events


In [ ]:
# Short execution check, not a performance result.
smoke_root = project / "runs/p2-jaah-kaggle-smoke"
if not smoke_root.exists():
    subprocess.run([sys.executable, '-m', 'acr.experiments.jaah.event',
                    "--output", str(smoke_root), "--smoke"], check=True)
assert (smoke_root / "comparison.json").exists(), "Incomplete smoke run; inspect it before retrying."


In [ ]:
assert torch.cuda.is_available(), "Enable a Kaggle GPU before the full pilot."
run_root = project / "runs/p2-jaah-pilot"
assert not run_root.exists(), "Use a fresh output path; existing results are protected."
subprocess.run([sys.executable, '-m', 'acr.experiments.jaah.event',
                "--output", str(run_root), "--device", "cuda"], check=True)


In [ ]:
print((run_root / "comparison.json").read_text())
for arm in ("billboard_control", "jaah_mixed"):
    print(arm)
    print((run_root / arm / "per_quality.csv").read_text())
    print((run_root / arm / "complementarity.csv").read_text())


In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/p2-jaah-pilot-results", "zip",
                              project, "runs/p2-jaah-pilot")
print(archive)


Save the executed notebook version, download p2-jaah-pilot-results.zip, and put it in the local runs/imports/ folder. The ZIP contains both jobs, checkpoints, logits, source exposure and F/E complementarity. Do not judge success from aggregate accuracy alone; inspect rare precision, support and common-chord regressions before repeating seeds. No JAAH/audio files are included in the result ZIP.
